# Error Handling in Function Calling for Agentic Workflows

This notebook demonstrates best practices for handling errors and invalid inputs in agentic workflows using function calling.

## Table of Contents
1. [Introduction](#introduction)
2. [Common Error Scenarios](#scenarios)
3. [Basic Error Handling](#basic)
4. [Advanced Error Handling Patterns](#advanced)
5. [Input Validation](#validation)
6. [Error Recovery Strategies](#recovery)
7. [Best Practices](#best-practices)

## 1. Introduction <a id="introduction"></a>

When building agentic workflows with function calling capabilities, robust error handling is crucial for:
- **Reliability**: Preventing system crashes from invalid inputs
- **User Experience**: Providing clear feedback on what went wrong
- **Debugging**: Making it easier to identify and fix issues
- **Security**: Protecting against malicious inputs

In [ ]:
import json
from typing import Any, Dict, List, Optional, Union, Callable
from enum import Enum
from dataclasses import dataclass
import traceback
from datetime import datetime

## 2. Common Error Scenarios <a id="scenarios"></a>

### Error Types in Function Calling:
1. **Missing Required Parameters**
2. **Invalid Parameter Types**
3. **Out-of-Range Values**
4. **Malformed JSON**
5. **Function Execution Errors**
6. **Resource Unavailability**

In [ ]:
class ErrorType(Enum):
    """Enumeration of error types in function calling"""
    MISSING_PARAMETER = "missing_parameter"
    INVALID_TYPE = "invalid_type"
    OUT_OF_RANGE = "out_of_range"
    MALFORMED_INPUT = "malformed_input"
    EXECUTION_ERROR = "execution_error"
    RESOURCE_ERROR = "resource_error"
    VALIDATION_ERROR = "validation_error"

@dataclass
class FunctionError:
    """Structured error information"""
    error_type: ErrorType
    message: str
    function_name: str
    parameter: Optional[str] = None
    timestamp: str = None
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now().isoformat()
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "error_type": self.error_type.value,
            "message": self.message,
            "function_name": self.function_name,
            "parameter": self.parameter,
            "timestamp": self.timestamp
        }
    
    def __str__(self):
        return f"[{self.error_type.value}] {self.function_name}: {self.message}"

## 3. Basic Error Handling <a id="basic"></a>

Let's start with basic error handling patterns for function calling.

In [ ]:
class FunctionResult:
    """Container for function execution results"""
    def __init__(self, success: bool, data: Any = None, error: Optional[FunctionError] = None):
        self.success = success
        self.data = data
        self.error = error
    
    def to_dict(self) -> Dict[str, Any]:
        result = {"success": self.success}
        if self.success:
            result["data"] = self.data
        else:
            result["error"] = self.error.to_dict() if self.error else None
        return result

def safe_function_call(func: Callable, func_name: str, **kwargs) -> FunctionResult:
    """
    Safely execute a function with error handling.
    
    Args:
        func: The function to execute
        func_name: Name of the function for error reporting
        **kwargs: Arguments to pass to the function
    
    Returns:
        FunctionResult with success status and data or error
    """
    try:
        result = func(**kwargs)
        return FunctionResult(success=True, data=result)
    except TypeError as e:
        error = FunctionError(
            error_type=ErrorType.INVALID_TYPE,
            message=str(e),
            function_name=func_name
        )
        return FunctionResult(success=False, error=error)
    except ValueError as e:
        error = FunctionError(
            error_type=ErrorType.OUT_OF_RANGE,
            message=str(e),
            function_name=func_name
        )
        return FunctionResult(success=False, error=error)
    except Exception as e:
        error = FunctionError(
            error_type=ErrorType.EXECUTION_ERROR,
            message=f"{type(e).__name__}: {str(e)}",
            function_name=func_name
        )
        return FunctionResult(success=False, error=error)

### Example: Basic Function with Error Handling

In [ ]:
def calculate_discount(price: float, discount_percent: float) -> Dict[str, float]:
    """
    Calculate discounted price.
    
    Args:
        price: Original price (must be positive)
        discount_percent: Discount percentage (0-100)
    
    Returns:
        Dictionary with original price, discount, and final price
    """
    if price < 0:
        raise ValueError("Price cannot be negative")
    if not 0 <= discount_percent <= 100:
        raise ValueError("Discount percent must be between 0 and 100")
    
    discount_amount = price * (discount_percent / 100)
    final_price = price - discount_amount
    
    return {
        "original_price": price,
        "discount_amount": discount_amount,
        "discount_percent": discount_percent,
        "final_price": final_price
    }

# Test with valid inputs
print("Valid input:")
result = safe_function_call(calculate_discount, "calculate_discount", price=100.0, discount_percent=20.0)
print(json.dumps(result.to_dict(), indent=2))

# Test with invalid inputs
print("\nInvalid input (negative price):")
result = safe_function_call(calculate_discount, "calculate_discount", price=-100.0, discount_percent=20.0)
print(json.dumps(result.to_dict(), indent=2))

print("\nInvalid input (out of range discount):")
result = safe_function_call(calculate_discount, "calculate_discount", price=100.0, discount_percent=150.0)
print(json.dumps(result.to_dict(), indent=2))

print("\nInvalid input (wrong type):")
result = safe_function_call(calculate_discount, "calculate_discount", price="hundred", discount_percent=20.0)
print(json.dumps(result.to_dict(), indent=2))

## 4. Advanced Error Handling Patterns <a id="advanced"></a>

### Parameter Validation Decorator

In [ ]:
from functools import wraps
from typing import get_type_hints

def validate_parameters(func):
    """
    Decorator to validate function parameters against type hints.
    """
    @wraps(func)
    def wrapper(*args, **kwargs):
        # Get type hints
        hints = get_type_hints(func)
        
        # Get function signature
        import inspect
        sig = inspect.signature(func)
        bound_args = sig.bind(*args, **kwargs)
        bound_args.apply_defaults()
        
        # Validate each parameter
        for param_name, param_value in bound_args.arguments.items():
            if param_name in hints and param_name != 'return':
                expected_type = hints[param_name]
                
                # Handle Optional types
                if hasattr(expected_type, '__origin__'):
                    if expected_type.__origin__ is Union:
                        types = expected_type.__args__
                        if not isinstance(param_value, types):
                            raise TypeError(
                                f"Parameter '{param_name}' expected one of {types}, "
                                f"got {type(param_value).__name__}"
                            )
                        continue
                
                # Basic type checking
                if not isinstance(param_value, expected_type):
                    raise TypeError(
                        f"Parameter '{param_name}' expected {expected_type.__name__}, "
                        f"got {type(param_value).__name__}"
                    )
        
        return func(*args, **kwargs)
    
    return wrapper

@validate_parameters
def send_email(to: str, subject: str, body: str, cc: Optional[List[str]] = None) -> Dict[str, Any]:
    """
    Simulate sending an email.
    
    Args:
        to: Recipient email address
        subject: Email subject
        body: Email body
        cc: Optional list of CC recipients
    
    Returns:
        Status of the email send operation
    """
    # Validate email format
    if '@' not in to:
        raise ValueError(f"Invalid email address: {to}")
    
    if cc:
        for email in cc:
            if '@' not in email:
                raise ValueError(f"Invalid CC email address: {email}")
    
    return {
        "status": "sent",
        "to": to,
        "subject": subject,
        "cc": cc or [],
        "timestamp": datetime.now().isoformat()
    }

# Test the decorated function
print("Valid email:")
result = safe_function_call(
    send_email, 
    "send_email",
    to="user@example.com",
    subject="Test",
    body="Hello!"
)
print(json.dumps(result.to_dict(), indent=2))

print("\nInvalid email (wrong type for 'to'):")
result = safe_function_call(
    send_email,
    "send_email",
    to=12345,  # Wrong type
    subject="Test",
    body="Hello!"
)
print(json.dumps(result.to_dict(), indent=2))

print("\nInvalid email format:")
result = safe_function_call(
    send_email,
    "send_email",
    to="invalid-email",
    subject="Test",
    body="Hello!"
)
print(json.dumps(result.to_dict(), indent=2))

## 5. Input Validation <a id="validation"></a>

### Schema-Based Validation

In [ ]:
class ParameterSchema:
    """Schema for parameter validation"""
    def __init__(self, param_type: type, required: bool = True, 
                 min_value: Any = None, max_value: Any = None,
                 allowed_values: List[Any] = None,
                 pattern: str = None):
        self.param_type = param_type
        self.required = required
        self.min_value = min_value
        self.max_value = max_value
        self.allowed_values = allowed_values
        self.pattern = pattern
    
    def validate(self, value: Any, param_name: str) -> Optional[str]:
        """Validate a value against this schema. Returns error message if invalid."""
        # Check type
        if not isinstance(value, self.param_type):
            return f"Expected {self.param_type.__name__}, got {type(value).__name__}"
        
        # Check range for numeric types
        if isinstance(value, (int, float)):
            if self.min_value is not None and value < self.min_value:
                return f"Value {value} is below minimum {self.min_value}"
            if self.max_value is not None and value > self.max_value:
                return f"Value {value} exceeds maximum {self.max_value}"
        
        # Check allowed values
        if self.allowed_values and value not in self.allowed_values:
            return f"Value {value} not in allowed values: {self.allowed_values}"
        
        # Check pattern for strings
        if self.pattern and isinstance(value, str):
            import re
            if not re.match(self.pattern, value):
                return f"Value '{value}' does not match pattern '{self.pattern}'"
        
        return None

class FunctionValidator:
    """Validator for function parameters"""
    def __init__(self, function_name: str, schemas: Dict[str, ParameterSchema]):
        self.function_name = function_name
        self.schemas = schemas
    
    def validate(self, params: Dict[str, Any]) -> List[FunctionError]:
        """Validate parameters against schemas. Returns list of errors."""
        errors = []
        
        # Check required parameters
        for param_name, schema in self.schemas.items():
            if schema.required and param_name not in params:
                errors.append(FunctionError(
                    error_type=ErrorType.MISSING_PARAMETER,
                    message=f"Required parameter '{param_name}' is missing",
                    function_name=self.function_name,
                    parameter=param_name
                ))
                continue
            
            # Validate parameter if present
            if param_name in params:
                error_msg = schema.validate(params[param_name], param_name)
                if error_msg:
                    errors.append(FunctionError(
                        error_type=ErrorType.VALIDATION_ERROR,
                        message=error_msg,
                        function_name=self.function_name,
                        parameter=param_name
                    ))
        
        return errors

# Example: Define a function with validation
create_user_validator = FunctionValidator(
    function_name="create_user",
    schemas={
        "username": ParameterSchema(
            param_type=str,
            required=True,
            pattern=r"^[a-zA-Z0-9_]{3,20}$"
        ),
        "email": ParameterSchema(
            param_type=str,
            required=True,
            pattern=r"^[\w\.-]+@[\w\.-]+\.\w+$"
        ),
        "age": ParameterSchema(
            param_type=int,
            required=True,
            min_value=0,
            max_value=150
        ),
        "role": ParameterSchema(
            param_type=str,
            required=False,
            allowed_values=["user", "admin", "moderator"]
        )
    }
)

# Test validation
print("Valid parameters:")
params = {
    "username": "john_doe",
    "email": "john@example.com",
    "age": 25,
    "role": "user"
}
errors = create_user_validator.validate(params)
print(f"Errors: {[str(e) for e in errors]}")

print("\nInvalid parameters:")
params = {
    "username": "jd",  # Too short
    "email": "invalid-email",  # Invalid format
    "age": 200,  # Out of range
    "role": "superuser"  # Not in allowed values
}
errors = create_user_validator.validate(params)
for error in errors:
    print(f"  - {error}")

print("\nMissing required parameters:")
params = {
    "username": "john_doe"
}
errors = create_user_validator.validate(params)
for error in errors:
    print(f"  - {error}")

## 6. Error Recovery Strategies <a id="recovery"></a>

### Retry Logic and Fallback Mechanisms

In [ ]:
import time
from typing import Tuple

class RetryConfig:
    """Configuration for retry logic"""
    def __init__(self, max_attempts: int = 3, delay: float = 1.0, 
                 backoff_factor: float = 2.0, retry_on: List[ErrorType] = None):
        self.max_attempts = max_attempts
        self.delay = delay
        self.backoff_factor = backoff_factor
        self.retry_on = retry_on or [ErrorType.RESOURCE_ERROR, ErrorType.EXECUTION_ERROR]

def execute_with_retry(func: Callable, func_name: str, 
                       retry_config: RetryConfig,
                       **kwargs) -> Tuple[FunctionResult, int]:
    """
    Execute a function with retry logic.
    
    Returns:
        Tuple of (FunctionResult, number of attempts made)
    """
    attempts = 0
    current_delay = retry_config.delay
    
    while attempts < retry_config.max_attempts:
        attempts += 1
        result = safe_function_call(func, func_name, **kwargs)
        
        if result.success:
            return result, attempts
        
        # Check if we should retry
        if result.error.error_type not in retry_config.retry_on:
            return result, attempts
        
        # Don't sleep on the last attempt
        if attempts < retry_config.max_attempts:
            print(f"Attempt {attempts} failed: {result.error.message}. Retrying in {current_delay}s...")
            time.sleep(current_delay)
            current_delay *= retry_config.backoff_factor
    
    return result, attempts

# Example: Unreliable function that might fail
call_count = 0

def unreliable_api_call(data: str) -> Dict[str, Any]:
    """Simulates an API call that might fail"""
    global call_count
    call_count += 1
    
    # Fail on first two attempts, succeed on third
    if call_count < 3:
        raise ConnectionError(f"Network error (attempt {call_count})")
    
    return {
        "status": "success",
        "data": data.upper(),
        "attempts": call_count
    }

# Test retry logic
print("Testing retry logic:")
call_count = 0
retry_config = RetryConfig(max_attempts=5, delay=0.1, backoff_factor=1.5)
result, attempts = execute_with_retry(
    unreliable_api_call,
    "unreliable_api_call",
    retry_config,
    data="hello world"
)
print(f"\nFinal result after {attempts} attempts:")
print(json.dumps(result.to_dict(), indent=2))

### Fallback Functions

In [ ]:
def execute_with_fallback(primary_func: Callable, 
                          fallback_func: Callable,
                          func_name: str,
                          **kwargs) -> FunctionResult:
    """
    Execute a function with a fallback if it fails.
    
    Args:
        primary_func: Primary function to execute
        fallback_func: Fallback function to use if primary fails
        func_name: Name for error reporting
        **kwargs: Arguments to pass to both functions
    
    Returns:
        FunctionResult from either primary or fallback
    """
    # Try primary function
    result = safe_function_call(primary_func, func_name, **kwargs)
    
    if result.success:
        return result
    
    # Try fallback
    print(f"Primary function failed: {result.error.message}")
    print("Attempting fallback...")
    
    fallback_result = safe_function_call(fallback_func, f"{func_name}_fallback", **kwargs)
    
    if fallback_result.success:
        # Add metadata indicating fallback was used
        if isinstance(fallback_result.data, dict):
            fallback_result.data["_fallback_used"] = True
            fallback_result.data["_primary_error"] = result.error.message
    
    return fallback_result

# Example: Primary and fallback functions
def get_data_from_cache(key: str) -> Dict[str, Any]:
    """Primary function: Get data from cache"""
    # Simulate cache miss
    raise KeyError(f"Key '{key}' not found in cache")

def get_data_from_database(key: str) -> Dict[str, Any]:
    """Fallback function: Get data from database"""
    # Simulate database query
    return {
        "key": key,
        "value": f"data_for_{key}",
        "source": "database"
    }

# Test fallback
print("Testing fallback mechanism:")
result = execute_with_fallback(
    get_data_from_cache,
    get_data_from_database,
    "get_data",
    key="user_123"
)
print("\nResult:")
print(json.dumps(result.to_dict(), indent=2))

## 7. Complete Example: Agent Function Executor <a id="best-practices"></a>

Putting it all together into a production-ready function executor.

In [ ]:
class AgentFunctionExecutor:
    """
    Production-ready function executor with comprehensive error handling.
    """
    def __init__(self):
        self.functions = {}
        self.validators = {}
        self.retry_configs = {}
        self.fallbacks = {}
        self.execution_log = []
    
    def register_function(self, name: str, func: Callable, 
                         validator: Optional[FunctionValidator] = None,
                         retry_config: Optional[RetryConfig] = None,
                         fallback: Optional[Callable] = None):
        """Register a function with optional validation, retry, and fallback."""
        self.functions[name] = func
        if validator:
            self.validators[name] = validator
        if retry_config:
            self.retry_configs[name] = retry_config
        if fallback:
            self.fallbacks[name] = fallback
    
    def execute(self, function_name: str, parameters: Dict[str, Any]) -> Dict[str, Any]:
        """
        Execute a registered function with full error handling.
        
        Returns:
            Dictionary with execution results and metadata
        """
        start_time = datetime.now()
        execution_record = {
            "function_name": function_name,
            "parameters": parameters,
            "start_time": start_time.isoformat(),
            "attempts": 0
        }
        
        # Check if function exists
        if function_name not in self.functions:
            error = FunctionError(
                error_type=ErrorType.EXECUTION_ERROR,
                message=f"Function '{function_name}' not found",
                function_name=function_name
            )
            execution_record["success"] = False
            execution_record["error"] = error.to_dict()
            self.execution_log.append(execution_record)
            return execution_record
        
        # Validate parameters
        if function_name in self.validators:
            validation_errors = self.validators[function_name].validate(parameters)
            if validation_errors:
                execution_record["success"] = False
                execution_record["validation_errors"] = [e.to_dict() for e in validation_errors]
                self.execution_log.append(execution_record)
                return execution_record
        
        func = self.functions[function_name]
        
        # Execute with retry if configured
        if function_name in self.retry_configs:
            result, attempts = execute_with_retry(
                func, function_name,
                self.retry_configs[function_name],
                **parameters
            )
            execution_record["attempts"] = attempts
        else:
            result = safe_function_call(func, function_name, **parameters)
            execution_record["attempts"] = 1
        
        # Try fallback if available and primary failed
        if not result.success and function_name in self.fallbacks:
            print(f"Primary function failed, trying fallback...")
            result = safe_function_call(
                self.fallbacks[function_name],
                f"{function_name}_fallback",
                **parameters
            )
            if result.success:
                execution_record["fallback_used"] = True
        
        # Record results
        end_time = datetime.now()
        execution_record["end_time"] = end_time.isoformat()
        execution_record["duration_ms"] = (end_time - start_time).total_seconds() * 1000
        execution_record["success"] = result.success
        
        if result.success:
            execution_record["result"] = result.data
        else:
            execution_record["error"] = result.error.to_dict()
        
        self.execution_log.append(execution_record)
        return execution_record
    
    def get_execution_log(self) -> List[Dict[str, Any]]:
        """Get the complete execution log."""
        return self.execution_log
    
    def get_statistics(self) -> Dict[str, Any]:
        """Get execution statistics."""
        total_executions = len(self.execution_log)
        successful = sum(1 for e in self.execution_log if e["success"])
        failed = total_executions - successful
        
        return {
            "total_executions": total_executions,
            "successful": successful,
            "failed": failed,
            "success_rate": successful / total_executions if total_executions > 0 else 0,
            "average_duration_ms": sum(e.get("duration_ms", 0) for e in self.execution_log) / total_executions if total_executions > 0 else 0
        }

### Demo: Complete Workflow

In [ ]:
# Initialize executor
executor = AgentFunctionExecutor()

# Define some functions
def process_order(order_id: str, quantity: int, priority: str) -> Dict[str, Any]:
    """Process an order"""
    if quantity <= 0:
        raise ValueError("Quantity must be positive")
    
    return {
        "order_id": order_id,
        "quantity": quantity,
        "priority": priority,
        "status": "processed",
        "timestamp": datetime.now().isoformat()
    }

def process_order_fallback(order_id: str, quantity: int, priority: str) -> Dict[str, Any]:
    """Fallback: Queue order for manual processing"""
    return {
        "order_id": order_id,
        "status": "queued_for_manual_processing",
        "reason": "automatic processing failed"
    }

# Register with validation
order_validator = FunctionValidator(
    function_name="process_order",
    schemas={
        "order_id": ParameterSchema(
            param_type=str,
            required=True,
            pattern=r"^ORD-\d{6}$"
        ),
        "quantity": ParameterSchema(
            param_type=int,
            required=True,
            min_value=1,
            max_value=1000
        ),
        "priority": ParameterSchema(
            param_type=str,
            required=True,
            allowed_values=["low", "medium", "high", "urgent"]
        )
    }
)

executor.register_function(
    "process_order",
    process_order,
    validator=order_validator,
    fallback=process_order_fallback
)

# Test various scenarios
print("=" * 60)
print("Test 1: Valid order")
print("=" * 60)
result = executor.execute("process_order", {
    "order_id": "ORD-123456",
    "quantity": 5,
    "priority": "high"
})
print(json.dumps(result, indent=2))

print("\n" + "=" * 60)
print("Test 2: Invalid order ID format")
print("=" * 60)
result = executor.execute("process_order", {
    "order_id": "INVALID",
    "quantity": 5,
    "priority": "high"
})
print(json.dumps(result, indent=2))

print("\n" + "=" * 60)
print("Test 3: Out of range quantity")
print("=" * 60)
result = executor.execute("process_order", {
    "order_id": "ORD-123456",
    "quantity": 2000,
    "priority": "high"
})
print(json.dumps(result, indent=2))

print("\n" + "=" * 60)
print("Test 4: Invalid priority")
print("=" * 60)
result = executor.execute("process_order", {
    "order_id": "ORD-123456",
    "quantity": 5,
    "priority": "critical"  # Not in allowed values
})
print(json.dumps(result, indent=2))

print("\n" + "=" * 60)
print("Test 5: Missing required parameter")
print("=" * 60)
result = executor.execute("process_order", {
    "order_id": "ORD-123456",
    "quantity": 5
    # Missing 'priority'
})
print(json.dumps(result, indent=2))

print("\n" + "=" * 60)
print("Execution Statistics")
print("=" * 60)
stats = executor.get_statistics()
print(json.dumps(stats, indent=2))

## Best Practices Summary

### 1. Always Validate Input
- Define clear schemas for all function parameters
- Validate types, ranges, and formats
- Return descriptive error messages

### 2. Use Structured Error Types
- Categorize errors (validation, execution, resource, etc.)
- Include context (function name, parameter name, timestamp)
- Make errors machine-readable and human-friendly

### 3. Implement Recovery Strategies
- Use retry logic for transient failures
- Provide fallback functions for critical operations
- Log all execution attempts for debugging

### 4. Handle Errors Gracefully
- Never expose raw exceptions to users
- Provide actionable error messages
- Fail fast for validation errors
- Retry for transient errors

### 5. Monitor and Log
- Keep execution logs with timestamps
- Track success/failure rates
- Monitor retry patterns
- Use logs for debugging and optimization

### 6. Test Error Scenarios
- Test with invalid inputs
- Test edge cases (min/max values, empty strings, etc.)
- Test resource failures
- Verify error messages are helpful

### 7. Design for User Experience
- Clear error messages guide users to fix issues
- Provide examples of valid inputs
- Suggest corrections when possible
- Make errors conversational for agent interactions

## Additional Example: AI Agent Integration

Here's how to integrate this error handling into an AI agent workflow:

In [ ]:
class AIAgent:
    """Simple AI agent that can call functions with error handling"""
    
    def __init__(self, executor: AgentFunctionExecutor):
        self.executor = executor
    
    def process_function_call(self, function_call: Dict[str, Any]) -> str:
        """
        Process a function call from the AI model.
        
        Args:
            function_call: Dictionary with 'name' and 'parameters'
        
        Returns:
            User-friendly response message
        """
        function_name = function_call.get("name")
        parameters = function_call.get("parameters", {})
        
        print(f"\nAgent: Calling function '{function_name}' with parameters: {parameters}")
        
        result = self.executor.execute(function_name, parameters)
        
        if result["success"]:
            return self._format_success_response(result)
        else:
            return self._format_error_response(result)
    
    def _format_success_response(self, result: Dict[str, Any]) -> str:
        """Format a successful result for the user"""
        data = result["result"]
        response = f"✓ Successfully executed '{result['function_name']}'\n"
        response += f"Result: {json.dumps(data, indent=2)}"
        
        if result.get("fallback_used"):
            response += "\n(Note: Fallback method was used)"
        
        return response
    
    def _format_error_response(self, result: Dict[str, Any]) -> str:
        """Format an error for the user with helpful guidance"""
        response = f"✗ Failed to execute '{result['function_name']}'\n\n"
        
        # Handle validation errors
        if "validation_errors" in result:
            response += "Validation errors:\n"
            for error in result["validation_errors"]:
                param = error.get("parameter", "unknown")
                message = error.get("message", "")
                response += f"  - {param}: {message}\n"
            response += "\nPlease check your inputs and try again."
        
        # Handle execution errors
        elif "error" in result:
            error = result["error"]
            response += f"Error: {error.get('message', 'Unknown error')}\n"
            response += f"Type: {error.get('error_type', 'unknown')}\n"
            
            if result.get("attempts", 0) > 1:
                response += f"\n(Attempted {result['attempts']} times)"
        
        return response

# Demo the AI agent
agent = AIAgent(executor)

print("=" * 60)
print("AI Agent Demo")
print("=" * 60)

# Successful call
print("\n" + "=" * 60)
print("Scenario 1: Valid function call")
print("=" * 60)
response = agent.process_function_call({
    "name": "process_order",
    "parameters": {
        "order_id": "ORD-789012",
        "quantity": 10,
        "priority": "urgent"
    }
})
print(f"\nResponse to user:\n{response}")

# Invalid call
print("\n" + "=" * 60)
print("Scenario 2: Invalid function call")
print("=" * 60)
response = agent.process_function_call({
    "name": "process_order",
    "parameters": {
        "order_id": "WRONG-FORMAT",
        "quantity": -5,
        "priority": "super-urgent"
    }
})
print(f"\nResponse to user:\n{response}")

## Conclusion

This notebook demonstrated comprehensive error handling strategies for function calling in agentic workflows:

1. **Structured error types** for categorizing and handling different failure modes
2. **Input validation** with schema-based parameter checking
3. **Retry logic** for handling transient failures
4. **Fallback mechanisms** for graceful degradation
5. **Execution logging** for monitoring and debugging
6. **User-friendly error messages** for AI agent interactions

By implementing these patterns, you can build robust agentic systems that handle errors gracefully and provide excellent user experience even when things go wrong.